# Chapter 4: Completeness and Canonical Models

This notebook follows Chapter 4 of [Boxes and Diamonds](https://bd.openlogicproject.org)
using the **Gamen.jl** package.

We cover:
- Subformulas and formula closure
- Derivability from sets of formulas (Definition 3.36)
- Consistency (Definition 3.39) and complete consistent sets (Definition 4.1)
- Lindenbaum's Lemma (Theorem 4.3)
- Modal operators on sets (Definition 4.5)
- Canonical model construction (Definition 4.11)
- The Truth Lemma (Proposition 4.12)
- Determination and completeness (Theorem 4.14, Corollary 4.15)
- Frame completeness (Theorem 4.16)

In [ ]:
using Pkg
Pkg.activate(joinpath(@__DIR__, "..", ".."))
using Gamen

In [ ]:
p = Atom(:p)
q = Atom(:q)
r = Atom(:r);

## Subformulas and Closure

Before constructing canonical models, we need tools for working with finite languages. The `subformulas` function collects all subformulas of a formula, and `formula_closure` extends a set to include negations.

In [ ]:
# All subformulas of □(p → q)
subformulas(Box(Implies(p, q)))

In [ ]:
# Closure adds negations — gives a finite "language" for canonical models
formula_closure([p, Box(p)])

## Derivability and Consistency

**Derivability from a set** (Definition 3.36): Γ ⊢\_Σ A means A follows from premises in Γ within system Σ. By soundness and completeness, we check this semantically.

**Consistency** (Definition 3.39): Γ is Σ-consistent iff ⊥ is not derivable from Γ.

In [ ]:
# Derivability: {p, p→q} ⊢_K q  (modus ponens)
println("{p, p→q} ⊢_K q:  ", is_derivable_from(SYSTEM_K, [p, Implies(p, q)], q; max_worlds=2))

# K proves □(p→p) but NOT □p→p
println("K ⊢ □(p→p):      ", is_derivable_from(SYSTEM_K, Formula[], Box(Implies(p, p)); max_worlds=2))
println("K ⊢ □p→p:        ", is_derivable_from(SYSTEM_K, Formula[], Implies(Box(p), p); max_worlds=2))

# KT proves □p→p (axiom T)
println("KT ⊢ □p→p:       ", is_derivable_from(SYSTEM_KT, Formula[], Implies(Box(p), p); max_worlds=2))

In [ ]:
# Consistency
println("{p, □p} K-consistent:     ", is_consistent(SYSTEM_K, [p, Box(p)]; max_worlds=2))
println("{p, ¬p} K-consistent:     ", is_consistent(SYSTEM_K, [p, Not(p)]; max_worlds=2))

# {□p, ¬p} is K-consistent (p fails at current world, true at all successors)
println("{□p, ¬p} K-consistent:    ", is_consistent(SYSTEM_K, [Box(p), Not(p)]; max_worlds=2))

# But KT-inconsistent (T says □p → p)
println("{□p, ¬p} KT-consistent:   ", is_consistent(SYSTEM_KT, [Box(p), Not(p)]; max_worlds=2))

## Complete Σ-Consistent Sets (Definition 4.1)

A set Γ is *complete Σ-consistent* if it is Σ-consistent and for every formula A in the language, either A ∈ Γ or ¬A ∈ Γ. These are the "maximally decided" consistent sets.

In [ ]:
lang_simple = formula_closure([p])  # {p, ¬p}

println("{p} complete w.r.t. {p, ¬p}:   ", is_complete_consistent(SYSTEM_K, [p], lang_simple; max_worlds=2))
println("{} complete w.r.t. {p, ¬p}:    ", is_complete_consistent(SYSTEM_K, Formula[], lang_simple; max_worlds=2))
println("{p, ¬p} complete w.r.t. {p, ¬p}: ", is_complete_consistent(SYSTEM_K, [p, Not(p)], lang_simple; max_worlds=2))

## Lindenbaum's Lemma (Theorem 4.3)

Every Σ-consistent set can be extended to a *complete* Σ-consistent set. The construction processes formulas one at a time: for each A, if adding A keeps the set consistent, add A; otherwise add ¬A.

In [ ]:
lang = formula_closure([p, Box(p)])
println("Language: ", [string(f) for f in lang])

# Extend {□p} to a complete K-consistent set
ext = lindenbaum_extend(SYSTEM_K, [Box(p)], lang; max_worlds=3)
println("Extension of {□p}: {", join(sort(string.(collect(ext))), ", "), "}")
println("□p ∈ extension: ", Box(p) ∈ ext)

## Modal Operators on Sets (Definition 4.5)

Definition 4.5 introduces operations on sets of formulas:
- □Γ = {□B : B ∈ Γ}
- ◇Γ = {◇B : B ∈ Γ}
- □⁻¹Γ = {B : □B ∈ Γ} — "unbox" the boxed formulas
- ◇⁻¹Γ = {B : ◇B ∈ Γ} — "undiamond" the diamond formulas

In [ ]:
Γ = Set{Formula}([Box(p), Box(q), Diamond(p), p])

println("Γ         = {", join(sort(string.(collect(Γ))), ", "), "}")
println("□Γ        = {", join(sort(string.(collect(box_set(Γ)))), ", "), "}")
println("◇Γ        = {", join(sort(string.(collect(diamond_set(Γ)))), ", "), "}")
println("□⁻¹Γ      = {", join(sort(string.(collect(box_inverse(Γ)))), ", "), "}")
println("◇⁻¹Γ      = {", join(sort(string.(collect(diamond_inverse(Γ)))), ", "), "}")

## Canonical Models (Definition 4.11)

The *canonical model* M^Σ = ⟨W^Σ, R^Σ, V^Σ⟩ where:

1. **W^Σ** = all complete Σ-consistent sets
2. **R^Σ** ΔΔ' iff □⁻¹Δ ⊆ Δ' (if □A ∈ Δ then A ∈ Δ')
3. **V^Σ**(p) = {Δ : p ∈ Δ}

For a finite language, we can enumerate all complete consistent sets and build this model explicitly.

In [ ]:
# Canonical model for K over {p, □p}
cm_k = canonical_model(SYSTEM_K, [p, Box(p)]; max_worlds=3)
println(cm_k)
println()

# Inspect the worlds and accessibility
for (i, Δ) in enumerate(cm_k.worlds)
    wname = Symbol("Δ", i)
    succs = accessible(cm_k.model.frame, wname)
    formulas = join(sort(string.(collect(Δ))), ", ")
    println("  Δ$i = {$formulas}")
    println("       sees: $succs")
end

## The Truth Lemma (Proposition 4.12)

The heart of the completeness proof: for every formula A and every world Δ,

**M^Σ, Δ ⊩ A  if and only if  A ∈ Δ**

This connects semantic satisfaction with set membership.

In [ ]:
println("Truth Lemma holds for K canonical model: ", truth_lemma_holds(cm_k))

## Frame Completeness (Theorem 4.16)

The canonical model's frame inherits properties from the axiom schemas. This extends completeness beyond K to all the standard systems.

| System contains... | Canonical model frame is... |
|:---|:---|
| D: □A → ◇A | serial |
| T: □A → A | reflexive |
| B: A → □◇A | symmetric |
| 4: □A → □□A | transitive |
| 5: ◇A → □◇A | euclidean |

In [ ]:
# KT canonical model is reflexive
cm_kt = canonical_model(SYSTEM_KT, [p, Box(p)]; max_worlds=3)
println("KT: ", length(cm_kt.worlds), " worlds")
println("  Truth Lemma: ", truth_lemma_holds(cm_kt))
println("  Reflexive:   ", is_reflexive(cm_kt.model.frame))

In [ ]:
# KD canonical model is serial
cm_kd = canonical_model(SYSTEM_KD, [p, Box(p)]; max_worlds=3)
println("KD: ", length(cm_kd.worlds), " worlds")
println("  Truth Lemma: ", truth_lemma_holds(cm_kd))
println("  Serial:      ", is_serial(cm_kd.model.frame))

In [ ]:
# S4 canonical model: reflexive AND transitive
# (need □□p in language for transitivity to manifest)
cm_s4 = canonical_model(SYSTEM_S4, [p, Box(p), Box(Box(p))]; max_worlds=3)
println("S4: ", length(cm_s4.worlds), " worlds")
println("  Truth Lemma: ", truth_lemma_holds(cm_s4))
println("  Reflexive:   ", is_reflexive(cm_s4.model.frame))
println("  Transitive:  ", is_transitive(cm_s4.model.frame))

## System Distinctness

Completeness lets us show systems are *distinct*: if a formula is derivable in one system but not another, they must differ.

In [ ]:
# Prop 3.32: KD ⊊ KT — Schema T is KT-derivable but not KD-derivable
schema_t = Implies(Box(p), p)
println("□p → p in KT: ", is_derivable_from(SYSTEM_KT, Formula[], schema_t; max_worlds=2))
println("□p → p in KD: ", is_derivable_from(SYSTEM_KD, Formula[], schema_t; max_worlds=2))

# Prop 3.33: KB ≠ K4 — Schema 4 is not KB-derivable
schema_4 = Implies(Box(p), Box(Box(p)))
println("□p → □□p in KB: ", is_derivable_from(SYSTEM_KB, Formula[], schema_4; max_worlds=2))
println("□p → □□p in K4: ", is_derivable_from(SYSTEM_K4, Formula[], schema_4; max_worlds=2))

## Determination (Definition 4.13)

A model M *determines* a system Σ if M ⊩ A iff Σ ⊢ A for every formula A. The canonical model determines its system — that's Theorem 4.14.

In [ ]:
println("K canonical model determines K: ", determines(cm_k.model, SYSTEM_K, [p]; max_worlds=3))

## Summary

Chapter 4 establishes the **completeness** of modal logics:

1. Every Σ-consistent set extends to a *complete* Σ-consistent set (Lindenbaum's Lemma)
2. The *canonical model* has these sets as worlds, with accessibility defined by □⁻¹Δ ⊆ Δ'
3. The *Truth Lemma* ensures M^Σ, Δ ⊩ A ↔ A ∈ Δ
4. Therefore every valid formula is derivable (**completeness**)
5. The canonical model's frame properties match the axiom schemas, extending completeness to KT, KD, S4, S5, etc.

In [ ]:
# Scratch space for exercises